In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
import numpy as np
import time
import os
from pathlib import Path
import json

In [2]:
df = pd.read_csv('anime_df.csv')
df.columns.values[0] = 'usernames'

In [3]:
df = df.set_index(df.columns[0])
df = df.astype(float)

In [4]:
conf_path = Path("../tauri.conf.json")

with open(conf_path, 'r') as f:
    config = json.load(f)

In [5]:
app_data_path = Path(os.getenv('APPDATA') or Path.home() / ".local/share")

# Path objects handle joining naturally with the / operator
watchlist_path = app_data_path / config['identifier'] / 'watchlist.json'

print(f"Watchlist path: {watchlist_path}")
print(f"DEBUG: Próbuję czytać z: {os.path.abspath(watchlist_path)}")

Watchlist path: C:\Users\mateu\AppData\Roaming\com.Mattisiek.animeChecker\watchlist.json
DEBUG: Próbuję czytać z: C:\Users\mateu\AppData\Roaming\com.Mattisiek.animeChecker\watchlist.json


In [6]:
def get_recommendations(user, df_orig, df_norm, n=5, k=10):
    # 1. KNN znajdzie k+1 sąsiadów (Ty + k innych)
    knn = NearestNeighbors(n_neighbors=k+1, metric='cosine')
    knn.fit(df_norm)
    
    # 2. Znalezienie sąsiadów dla Current_User
    user_idx = df_norm.index.get_loc(user)
    distances, indices = knn.kneighbors(df_norm.iloc[[user_idx]])
    
    # Podobieństwo to 1 - dystans
    similarities = 1 - distances.flatten()
    neighbor_indices = indices.flatten()
    
    # Tworzymy Series z podobieństwami, pomijając siebie (indeks 0)
    # weights to teraz nasze wagi do średniej ważonej
    similar_users = pd.Series(similarities[1:], index=df_norm.index[neighbor_indices[1:]])
    
    # 3. Średnia ocen Current_User
    user_ratings = df_orig.loc[user]
    user_mean = user_ratings[user_ratings != 0].mean() if (user_ratings != 0).any() else 0
    
    # 4. Optymalizacja: Sprawdzaj tylko anime, które widzieli sąsiedzi (candidate_anime)
    # Zamiast pętli po wszystkich anime w bazie
    neighbor_ratings = df_orig.loc[similar_users.index]
    candidate_mask = (neighbor_ratings > 0).any(axis=0) & (user_ratings == 0)
    candidate_anime = neighbor_ratings.columns[candidate_mask]
    
    predictions = {}
    for anime in candidate_anime:
        # Wybierz sąsiadów, którzy widzieli to konkretne anime
        relevant_indices = similar_users.index[neighbor_ratings[anime] > 0]
        
        if len(relevant_indices) > 0:
            # WAGI bierzemy bezpośrednio z similar_users (wynik KNN)
            weights = similar_users.loc[relevant_indices]
            norm_ratings = df_norm.loc[relevant_indices, anime]
            
            if weights.sum() > 0: pass
            pred_deviation = np.average(norm_ratings, weights=weights)
            predictions[anime] = user_mean + pred_deviation
    
    if not predictions:
        return {}
        
    recommendations = pd.Series(predictions).sort_values(ascending=False)
    return recommendations.head(n).to_dict() # .to_dict() ułatwi JS odczyt

In [7]:
t0 = time.time()

In [8]:
user_data = pd.read_json(watchlist_path)

# Konwersja na listę słowników
user_data = user_data.to_dict(orient='records')

In [9]:
user_ratings = {
    #f"{entry['mal_id']}": f"{entry['score']}"
    f"{entry['mal_id']}_{entry['title']}": 7.0
    for entry in user_data
}

# print(user_ratings)

In [10]:
new_user_name = "Current_User"

new_user_row = pd.Series(0, index=df.columns, name=new_user_name)

for anime, rating in user_ratings.items():
    if anime in new_user_row.index:
        new_user_row[anime] = rating


df_extended = pd.concat([df, new_user_row.to_frame().T])
df_normalized = df_extended.apply(lambda row: row - row[row != 0].mean() if (row != 0).any() else row, axis=1)

In [11]:
recommendations = get_recommendations('Current_User', df_extended, df_normalized, n=5, k=5)

for anime_key, score in recommendations.items():
    id_ref, name = anime_key.split('_', 1)
    print(f"Recommend: {name} (ID: {id_ref}) with predicted score: {score:.2f}")

Recommend: Naruto: Shippuuden (ID: 1735) with predicted score: 8.44
Recommend: Kiseijuu: Sei no Kakuritsu (ID: 22535) with predicted score: 8.44
Recommend: Gyakuten Saiban: Sono "Shinjitsu", Igi Ari! (ID: 31630) with predicted score: 8.37
Recommend: Black Clover (ID: 34572) with predicted score: 8.37
Recommend: Gyakuten Saiban: Sono "Shinjitsu", Igi Ari! Season 2 (ID: 37490) with predicted score: 8.37


In [12]:
print(time.time() - t0)

0.9807090759277344
